# Group 2: Stock Factor Betas (stagger 2)

Estimate each stock's currency, market, and sector betas on stagger 2, then build the
covariance matrix of the residual returns.

In [1]:
from pathlib import Path

import pandas as pd
import statsmodels.api as sm

DATA_FILE = Path("../data/companies_and_staggered_returns.xlsx")
STAGGER = "stagger_2"

FX = ["MXN", "SGD", "CAD", "DKK", "EUR", "AUD"]
SECTORS = ["XLB", "XLC", "XLE", "XLF", "XLI", "XLK", "XLP", "XLRE", "XLU", "XLV", "XLY"]


## Load the data

Get the stock list from the `Companies` sheet, and load stagger 2's time-weighted
returns and residual sector factors. CAD/USD is the corrected series.

In [2]:
companies = pd.read_excel(DATA_FILE, sheet_name="Companies")

# Keep only real stocks: one row per ticker, and no ETF / FX / SPY rows
stocks = (
    companies
    .dropna(subset=["sector"])
    .drop_duplicates(subset=["ticker"])
    .reset_index(drop=True)
)

returns = pd.read_excel(
    DATA_FILE, sheet_name=STAGGER, skiprows=254, nrows=80, parse_dates=["date"]
).set_index("date")
sector_factors = pd.read_excel(
    DATA_FILE, sheet_name=STAGGER, skiprows=338, nrows=80, parse_dates=["date"]
).set_index("date")

sector_to_etf = {
    "Communication Services": "XLC",
    "Consumer Discretionary": "XLY",
    "Consumer Staples": "XLP",
    "Energy": "XLE",
    "Financials": "XLF",
    "Health Care": "XLV",
    "Industrials": "XLI",
    "Materials": "XLB",
    "Real Estate": "XLRE",
    "Technology": "XLK",
    "Utilities": "XLU",
}

print(f"{len(stocks)} stocks")
print(f"Foreign stocks: {stocks.loc[stocks['currency'] != 'USD', 'ticker'].tolist()}")


43 stocks
Foreign stocks: ['VIST', 'STX', 'NTR', 'ASND', 'NOK']


## Regression helpers

`regress` is one factor with an intercept. `drop_weak_factors` is a multiple regression:
each round it drops the factor whose t-stat is closest to zero, if that t-stat is
between -2 and 2, and then refits. A dropped factor gets a beta of 0. The residual
carried forward is `return - (alpha + beta * factor)`.

In [3]:
def regress(y, x):
    """Regress y on one factor. Return beta, alpha, and the t-stat."""
    X = sm.add_constant(x)
    model = sm.OLS(y, X, missing="drop").fit()
    return model.params.iloc[1], model.params.iloc[0], model.tvalues.iloc[1]


def drop_weak_factors(y, factors):
    """Drop factors with -2 < t < 2, one at a time, and return the updated residual and betas."""
    names = list(factors.columns)
    betas = {name: 0.0 for name in names}

    while names:
        X = sm.add_constant(factors[names])
        model = sm.OLS(y, X, missing="drop").fit()
        t_stats = model.tvalues.iloc[1:]
        weakest = t_stats.abs().idxmin()

        if abs(t_stats[weakest]) < 2:
            names.remove(weakest)
            continue

        alpha = model.params.iloc[0]
        fitted = alpha
        for name in names:
            betas[name] = model.params[name]
            fitted = fitted + betas[name] * factors[name]
        y = y - fitted
        break

    return y, betas


## Steps 1 to 4

For each stock: (1) a foreign stock is regressed on its own currency and that beta is
kept, US stocks skip this; (2) the residual is regressed on the other currencies and
weak ones are dropped; (3) the residual is regressed on SPY and that beta is kept;
(4) the same two-step rule is used for the 11 sector factors, and every stock keeps
its own sector beta. What is left is the stock's residual return.

In [4]:
beta_rows = []
stock_specific = pd.DataFrame(index=returns.index)

for _, stock in stocks.iterrows():
    ticker = stock["ticker"]
    r = returns[ticker].astype(float)
    fx_betas = {name: 0.0 for name in FX}

    # Step 1: own currency, kept. US stocks skip this.
    if stock["currency"] != "USD":
        own_fx = stock["currency"]
        fx = returns[own_fx + "/USD"].astype(float)
        beta, alpha, _ = regress(r, fx)
        fx_betas[own_fx] = beta
        r = r - (alpha + beta * fx)
        other_fx = [name for name in FX if name != own_fx]
    else:
        other_fx = list(FX)

    # Step 2: multiple regression on the other currencies. Drop weak factors.
    fx_frame = pd.DataFrame(
        {name: returns[name + "/USD"].astype(float) for name in other_fx}
    )
    r, other_fx_betas = drop_weak_factors(r, fx_frame)
    fx_betas.update(other_fx_betas)

    # Step 3: market, kept.
    spy = returns["SPY"].astype(float)
    beta_mkt, alpha, _ = regress(r, spy)
    r = r - (alpha + beta_mkt * spy)

    # Step 4: own sector, kept, then the other sector factors.
    etf = sector_to_etf[stock["sector"]]
    sector_betas = {name: 0.0 for name in SECTORS}
    own_sector = sector_factors[etf + "_residual"].astype(float)
    beta, alpha, _ = regress(r, own_sector)
    sector_betas[etf] = beta
    r = r - (alpha + beta * own_sector)

    other_sectors = [name for name in SECTORS if name != etf]
    sector_frame = pd.DataFrame(
        {name: sector_factors[name + "_residual"].astype(float) for name in other_sectors}
    )
    r, other_sector_betas = drop_weak_factors(r, sector_frame)
    sector_betas.update(other_sector_betas)

    stock_specific[ticker] = r
    row = {
        "ticker": ticker,
        "sector": stock["sector"],
        "currency": stock["currency"],
    }
    for name in FX:
        row["beta_" + name] = fx_betas[name]
    row["beta_mkt"] = beta_mkt
    for name in SECTORS:
        row["beta_" + name] = sector_betas[name]
    beta_rows.append(row)

betas = pd.DataFrame(beta_rows)
print(f"{len(betas)} betas estimated on {STAGGER}")
for name in FX:
    kept = (betas["beta_" + name] != 0).sum()
    print(f"{name} beta kept for {kept} of {len(betas)} stocks")
betas.round(3)


43 betas estimated on stagger_2
MXN beta kept for 8 of 43 stocks
SGD beta kept for 12 of 43 stocks
CAD beta kept for 7 of 43 stocks
DKK beta kept for 13 of 43 stocks
EUR beta kept for 11 of 43 stocks
AUD beta kept for 21 of 43 stocks


,ticker,sector,currency,beta_MXN,beta_SGD,beta_CAD,beta_DKK,beta_EUR,beta_AUD,beta_mkt,...,beta_XLC,beta_XLE,beta_XLF,beta_XLI,beta_XLK,beta_XLP,beta_XLRE,beta_XLU,beta_XLV,beta_XLY
0,VST,Utilities,USD,0.000,4.010,0.000,-3.158,0.000,0.000,0.976,...,0.000,0.000,0.000,0.000,0.000,0.000,-0.820,0.794,-0.992,0.000
1,VIST,Energy,MXN,0.766,0.000,0.000,0.000,0.000,1.058,-0.027,...,0.000,0.835,-0.783,0.000,0.000,0.000,0.000,0.000,0.000,0.000
2,AEP,Utilities,USD,0.000,0.000,0.000,-6.552,6.577,0.000,-0.057,...,-0.234,0.000,0.000,0.000,0.000,0.000,0.000,1.021,0.000,0.000
3,ETR,Utilities,USD,0.000,0.000,0.000,-6.836,6.616,0.000,0.161,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.896,0.000,0.000
4,DIS,Communication Services,USD,0.000,0.000,0.000,0.000,0.000,1.183,0.762,...,0.614,0.000,0.701,0.000,0.000,0.000,-0.762,0.000,0.000,0.513
5,CMG,Consumer Discretionary,USD,0.000,0.000,0.000,0.000,0.000,0.000,0.749,...,0.000,0.000,0.000,0.000,0.000,0.734,0.000,0.000,0.000,1.064
6,AMZN,Consumer Discretionary,USD,0.000,0.000,0.000,0.000,0.000,1.438,1.042,...,0.000,0.000,0.000,0.000,0.000,-1.041,0.000,0.000,0.000,1.230
7,FIX,Industrials,USD,1.492,0.000,0.000,-2.234,0.000,2.332,0.625,...,-0.780,-0.311,0.000,1.013,0.000,0.000,-0.785,0.580,-1.042,0.000
8,STX,Technology,SGD,1.759,4.057,0.000,0.000,-3.342,2.423,0.942,...,0.000,0.000,0.000,0.000,2.088,0.000,0.000,0.000,0.000,0.000
9,VLO,Energy,USD,1.369,0.000,0.000,-2.140,0.000,1.586,0.127,...,0.000,1.017,0.000,0.000,0.000,-0.666,0.000,0.000,0.490,0.000


## Residual covariance matrix

The covariance of the 43 residual return series. The matrix equals its transpose.

In [5]:
cov = stock_specific.cov()
gap = (cov - cov.T).abs().to_numpy().max()
print(f"covariance matrix shape: {cov.shape}")
print(f"largest difference from its transpose: {gap:.3e}")
cov.round(4)


covariance matrix shape: (43, 43)
largest difference from its transpose: 0.000e+00


,VST,VIST,AEP,ETR,DIS,CMG,AMZN,FIX,STX,VLO,...,LLY,WELL,NOK,MSFT,GOOGL,COST,HD,KO,JNJ,BRK-B
VST,0.0103,0.0015,-0.0009,-0.0002,0.0001,0.0026,-0.0009,0.0019,0.0001,0.0003,...,0.0013,0.0002,-0.0021,-0.0003,-0.0012,0.0007,0.0008,-0.0006,-0.0007,0.0000
VIST,0.0015,0.0118,-0.0001,0.0001,0.0003,-0.0002,0.0010,0.0010,-0.0006,-0.0001,...,0.0019,0.0000,0.0015,-0.0005,-0.0005,0.0013,-0.0007,0.0003,-0.0002,0.0001
AEP,-0.0009,-0.0001,0.0009,0.0004,-0.0001,-0.0010,0.0005,0.0003,0.0005,0.0002,...,-0.0001,0.0000,0.0008,-0.0002,0.0003,-0.0000,-0.0000,0.0001,0.0000,0.0002
ETR,-0.0002,0.0001,0.0004,0.0012,-0.0001,-0.0008,0.0002,0.0008,0.0014,0.0003,...,-0.0003,0.0003,0.0008,-0.0003,0.0005,0.0004,0.0000,0.0002,0.0003,0.0001
DIS,0.0001,0.0003,-0.0001,-0.0001,0.0036,0.0016,0.0010,-0.0004,0.0003,0.0005,...,0.0001,0.0000,0.0001,0.0010,-0.0002,-0.0002,-0.0003,0.0001,-0.0002,-0.0001
CMG,0.0026,-0.0002,-0.0010,-0.0008,0.0016,0.0077,0.0004,-0.0005,-0.0025,-0.0011,...,0.0007,-0.0005,-0.0020,0.0008,-0.0009,0.0006,0.0001,-0.0001,-0.0010,-0.0004
AMZN,-0.0009,0.0010,0.0005,0.0002,0.0010,0.0004,0.0029,0.0007,0.0005,0.0003,...,0.0003,0.0004,0.0013,0.0007,0.0007,0.0005,-0.0001,0.0003,-0.0004,0.0003
FIX,0.0019,0.0010,0.0003,0.0008,-0.0004,-0.0005,0.0007,0.0072,0.0036,0.0009,...,0.0012,0.0003,0.0017,0.0000,0.0009,0.0007,0.0003,-0.0002,0.0001,0.0004
STX,0.0001,-0.0006,0.0005,0.0014,0.0003,-0.0025,0.0005,0.0036,0.0145,0.0022,...,-0.0003,0.0009,0.0034,-0.0026,0.0033,0.0006,0.0008,0.0004,0.0011,-0.0003
VLO,0.0003,-0.0001,0.0002,0.0003,0.0005,-0.0011,0.0003,0.0009,0.0022,0.0037,...,-0.0004,0.0002,0.0003,0.0004,-0.0001,0.0001,0.0003,0.0004,-0.0000,0.0002


## Save the results

Write the betas, the residual returns, and the covariance matrix to one Excel file.

In [6]:
output_file = Path("group2_stock_betas.xlsx")

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    betas.to_excel(writer, sheet_name="Betas", index=False)
    (stock_specific
     .rename_axis("date")
     .reset_index()
     .to_excel(writer, sheet_name="Residuals", index=False))
    (cov
     .rename_axis("ticker")
     .reset_index()
     .to_excel(writer, sheet_name="Covariance", index=False))

print(f"saved {output_file.resolve().name}")


saved group2_stock_betas.xlsx
